In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# Reuse/cache controls — immediately after Drive mount.
REUSE_PREDICTIONS = True
FOLDS = [0, 1, 2, 3, 4]
VESSELS = ["RCA", "LAD"]
MODEL_ZIP = "/content/drive/MyDrive/OpenPlaque/models/Dataset001_CCTA_DHM-20260703T233210Z-3-001.zip"
STUDY_ZIP = "/content/drive/MyDrive/OpenPlaque/Full_DICOM.zip"
OUT_ROOT = "/content/drive/MyDrive/OpenPlaque/GPU_Plaque_5Fold_Ensemble_v1"
SERIES = {"RCA": 1035, "LAD": 1043}


# OpenPlaque — RCA/LAD 5-fold plaque ensemble + uncertainty

Runs the existing Dataset001_CCTA_DHM nnU-Net model **fold-by-fold** on the validated RCA and LAD curved CCTA series. It preserves each fold mask, builds a 5-fold consensus, and writes voxelwise disagreement maps. This intentionally spends GPU compute on uncertainty rather than rerunning one opaque ensemble.

Research use only.

In [ ]:
!pip -q install nnunetv2 SimpleITK pydicom pandas matplotlib
import os, sys, shutil, zipfile, subprocess, json
from pathlib import Path
import numpy as np, pandas as pd, SimpleITK as sitk, matplotlib.pyplot as plt

repo = Path('/content/OpenPlaque')
if repo.exists(): shutil.rmtree(repo)
!git clone -q --depth 1 --branch source-plaque-ensemble-from-main https://github.com/pazzani/OpenPlaque.git /content/OpenPlaque
sys.path.insert(0, '/content/OpenPlaque/src')
!nvidia-smi


In [ ]:
# Configure nnU-Net and extract the existing trained 5-fold model locally.
os.environ["nnUNet_raw"] = "/content/nnUNet_raw"
os.environ["nnUNet_preprocessed"] = "/content/nnUNet_preprocessed"
os.environ["nnUNet_results"] = "/content/nnUNet_results"
for p in [os.environ["nnUNet_raw"], os.environ["nnUNet_preprocessed"], os.environ["nnUNet_results"]]:
    Path(p).mkdir(parents=True, exist_ok=True)

model_zip = Path(MODEL_ZIP)
if not model_zip.exists():
    raise FileNotFoundError(model_zip)
if not any(Path(os.environ["nnUNet_results"]).rglob("checkpoint_final.pth")):
    with zipfile.ZipFile(model_zip) as z:
        z.extractall(os.environ["nnUNet_results"])

print("Checkpoints:")
for p in sorted(Path(os.environ["nnUNet_results"]).rglob("checkpoint_final.pth")):
    print(" ", p)


In [ ]:
# Load the known UCLA curved coronary CCTA series from the full study.
from openplaque.study import OpenPlaqueStudy
study = OpenPlaqueStudy(STUDY_ZIP, extract_root="/content/full_dicom_gpu_plaque")
cases = {}
for vessel in VESSELS:
    image, volume, _ = study.load_series(SERIES[vessel])
    cases[vessel] = (image, volume)
    print(vessel, volume.shape, image.GetSpacing())


In [ ]:
# Export inputs once.
work = Path("/content/plaque_ensemble")
work.mkdir(exist_ok=True)
for vessel, (image, _) in cases.items():
    inp = work / vessel / "input"
    inp.mkdir(parents=True, exist_ok=True)
    sitk.WriteImage(image, str(inp / f"{vessel}_0000.nii.gz"))


In [ ]:
# Run all five trained folds independently. Each completed fold is cached on Drive.
out_root = Path(OUT_ROOT)
out_root.mkdir(parents=True, exist_ok=True)

def predict_fold(vessel, fold):
    inp = work / vessel / "input"
    final_dir = out_root / vessel / f"fold_{fold}"
    final_mask = final_dir / f"{vessel}.nii.gz"
    if REUSE_PREDICTIONS and final_mask.exists():
        print("reuse", vessel, "fold", fold)
        return final_mask
    local = work / vessel / f"fold_{fold}"
    if local.exists(): shutil.rmtree(local)
    local.mkdir(parents=True)
    cmd = ["nnUNetv2_predict", "-i", str(inp), "-o", str(local),
           "-d", "Dataset001_CCTA_DHM", "-c", "3d_fullres", "-f", str(fold)]
    print("RUN:", " ".join(cmd))
    subprocess.run(cmd, check=True)
    final_dir.mkdir(parents=True, exist_ok=True)
    shutil.copy2(local / f"{vessel}.nii.gz", final_mask)
    return final_mask

fold_masks = {}
for vessel in VESSELS:
    fold_masks[vessel] = [predict_fold(vessel, f) for f in FOLDS]
print("All fold predictions complete.")


In [ ]:
# Build hard-label consensus and a voxelwise fold-disagreement map.
summary_rows = []
for vessel in VESSELS:
    imgs = [sitk.ReadImage(str(p)) for p in fold_masks[vessel]]
    arr = np.stack([sitk.GetArrayFromImage(x).astype(np.uint8) for x in imgs], axis=0)
    n = arr.shape[0]
    counts = np.stack([(arr == lab).sum(axis=0) for lab in (0,1,2)], axis=0)
    consensus = np.argmax(counts, axis=0).astype(np.uint8)
    max_agree = counts.max(axis=0).astype(np.float32) / float(n)
    disagreement = 1.0 - max_agree
    
    vout = out_root / vessel
    cimg = sitk.GetImageFromArray(consensus); cimg.CopyInformation(imgs[0])
    dimg = sitk.GetImageFromArray(disagreement); dimg.CopyInformation(imgs[0])
    sitk.WriteImage(cimg, str(vout / f"{vessel}_5fold_consensus.nii.gz"))
    sitk.WriteImage(dimg, str(vout / f"{vessel}_5fold_disagreement.nii.gz"))
    
    voxel_mm3 = float(np.prod(imgs[0].GetSpacing()))
    for f, a in zip(FOLDS, arr):
        summary_rows.append({"vessel":vessel,"fold":f,
            "plaque_voxels":int((a==2).sum()),
            "plaque_volume_mm3":float((a==2).sum()*voxel_mm3)})
    summary_rows.append({"vessel":vessel,"fold":"consensus",
        "plaque_voxels":int((consensus==2).sum()),
        "plaque_volume_mm3":float((consensus==2).sum()*voxel_mm3),
        "mean_disagreement":float(disagreement.mean()),
        "plaque_region_mean_disagreement":float(disagreement[consensus==2].mean()) if np.any(consensus==2) else None,
        "fraction_voxels_with_fold_disagreement":float((disagreement>0).mean())})
    
summary = pd.DataFrame(summary_rows)
summary.to_csv(out_root/"plaque_5fold_summary.csv", index=False)
display(summary)


In [ ]:
# QC visualization: slice with the most consensus plaque for each artery.
fig, axes = plt.subplots(len(VESSELS), 3, figsize=(13, 5*len(VESSELS)))
axes = np.atleast_2d(axes)
for r, vessel in enumerate(VESSELS):
    vol = cases[vessel][1]
    cons = sitk.GetArrayFromImage(sitk.ReadImage(str(out_root/vessel/f"{vessel}_5fold_consensus.nii.gz")))
    dis = sitk.GetArrayFromImage(sitk.ReadImage(str(out_root/vessel/f"{vessel}_5fold_disagreement.nii.gz")))
    z = int(np.argmax((cons==2).sum(axis=(1,2))))
    axes[r,0].imshow(vol[z], cmap="gray", vmin=-200, vmax=800); axes[r,0].set_title(f"{vessel} CCTA slice {z}")
    axes[r,1].imshow(vol[z], cmap="gray", vmin=-200, vmax=800); axes[r,1].imshow(cons[z]==2, alpha=.55); axes[r,1].set_title("5-fold consensus plaque")
    axes[r,2].imshow(dis[z], vmin=0, vmax=.8); axes[r,2].set_title("fold disagreement")
    for ax in axes[r]: ax.axis("off")
fig.tight_layout()
fig.savefig(out_root/"plaque_ensemble_qc.png", dpi=180)
plt.show()


In [ ]:
# Package a compact report (fold masks stay in their Drive folders).
import html, datetime, zipfile
report = out_root/"OPENPLAQUE_GPU_PLAQUE_5FOLD_ENSEMBLE_REPORT.html"
report.write_text("<html><body><h1>OpenPlaque GPU plaque 5-fold ensemble</h1>"
                  "<p>Research use only.</p>"+summary.to_html(index=False)+
                  "<p><img src='plaque_ensemble_qc.png' style='max-width:100%'></p></body></html>")
meta = {
    "status":"COMPLETE",
    "model":"Dataset001_CCTA_DHM",
    "folds":FOLDS,
    "vessels":VESSELS,
    "series":SERIES,
    "outputs":"individual fold masks + hard-label consensus + disagreement map"
}
(out_root/"summary.json").write_text(json.dumps(meta, indent=2))
zip_path = out_root/"OPENPLAQUE_GPU_PLAQUE_5FOLD_ENSEMBLE_REPORT_BACK.zip"
with zipfile.ZipFile(zip_path,"w",zipfile.ZIP_DEFLATED) as z:
    for p in [report,out_root/"summary.json",out_root/"plaque_5fold_summary.csv",out_root/"plaque_ensemble_qc.png"]:
        z.write(p,p.name)
print("FINAL:", zip_path)
